# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas scikit-learn numpy Faker matplotlib seaborn
    print('Running in Colab')

# Cluster Analysis In-Depth

Cluster analysis is an unsupervised machine learning technique used to group similar data points together. In social science, this is often used for **segmentation**—identifying distinct subgroups within a population (e.g., voters, consumers, survey respondents) without predefined labels.

In this notebook, we will:
1.  Generate a realistic synthetic dataset of **Online Shoppers** (Age, Income, Spending Score).
2.  Preprocess the data (Standardization).
3.  Determine the optimal number of clusters using the **Elbow Method** and **Silhouette Analysis**.
4.  Perform **K-Means Clustering**.
5.  Visualize and interpret the resulting segments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

fake = Faker()
Faker.seed(42)
np.random.seed(42)

sns.set_style('whitegrid')

## 1. Data Generation

We will simulate a dataset of 500 customers with three key variables:
*   **Age**: 18-70
*   **Annual Income (k$)**: 15-150
*   **Spending Score (1-100)**: A metric assigned by the store based on customer behavior.

To make the clustering interesting, we will inject some structure by generating data around specific 'centers' but with noise.

In [ ]:
n_samples = 500

# Define 5 distinct groups (centroids) for Age, Income, Score
# Group 1: Young, Low Income, High Spend
# Group 2: Middle Age, Medium Income, Medium Spend
# Group 3: Elderly, Fixed Income, Low Spend
# Group 4: High Income, High Spend
# Group 5: High Income, Low Spend

groups = [
    {'n': 100, 'age': (18, 25), 'inc': (15, 40), 'score': (70, 99)},
    {'n': 150, 'age': (30, 50), 'inc': (40, 80), 'score': (40, 60)},
    {'n': 100, 'age': (60, 75), 'inc': (20, 50), 'score': (5, 30)},
    {'n': 75,  'age': (25, 45), 'inc': (70, 130), 'score': (75, 95)},
    {'n': 75,  'age': (35, 55), 'inc': (70, 130), 'score': (5, 25)}
]

data = []
for g in groups:
    for _ in range(g['n']):
        data.append({
            'Customer_ID': fake.uuid4(),
            'Age': np.random.randint(g['age'][0], g['age'][1]),
            'Annual_Income_k': np.random.randint(g['inc'][0], g['inc'][1]),
            'Spending_Score': np.random.randint(g['score'][0], g['score'][1])
        })

df = pd.DataFrame(data)
print("Dataset Shape:", df.shape)
print(df.head())

## 2. Preprocessing

K-Means is distance-based, so it is crucial to scale variables if they have different units (e.g., Age in years vs Income in thousands). However, in this specific common example, sometimes raw values are used for direct interpretability. For best practice, we will use `StandardScaler`.

In [ ]:
features = ['Age', 'Annual_Income_k', 'Spending_Score']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled Data (First 5 rows):\n", X_scaled[:5])

## 3. Determining Number of Clusters (k)

### The Elbow Method
We calculate the **Inertia** (Sum of Squared Errors) for different values of $k$. We look for the "elbow" point where adding more clusters yields diminishing returns.

In [ ]:
inertia = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(k_range, inertia, marker='o', linestyle='--')
plt.title('Elbow Method')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.show()

### Silhouette Score
The Silhouette Score measures how similar a point is to its own cluster compared to other clusters. A higher score is better. We generated 5 distinct groups, so we expect $k=5$ to perform well.

In [ ]:
sil_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)

plt.figure(figsize=(10, 5))
plt.bar(range(2, 11), sil_scores, color='skyblue')
plt.title('Silhouette Scores')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Score')
plt.show()

## 4. K-Means Clustering

Based on the analysis above, we choose $k=5$.

In [ ]:
k_optimal = 5
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Inverse transform centers for interpretation
centers = scaler.inverse_transform(kmeans.cluster_centers_)
df_centers = pd.DataFrame(centers, columns=features)
df_centers['Cluster'] = range(k_optimal)

print("Cluster Centers (Original Scale):")
print(df_centers)

## 5. Visualization

Since we have 3 dimensions, we can visualize the clusters using pairwise scatterplots.

In [ ]:
plt.figure(figsize=(15, 6))

# Plot 1: Income vs Spending
plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x='Annual_Income_k', y='Spending_Score', hue='Cluster', palette='viridis', s=60)
plt.title('Income vs Spending Score')

# Plot 2: Age vs Spending
plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='Age', y='Spending_Score', hue='Cluster', palette='viridis', s=60)
plt.title('Age vs Spending Score')

plt.show()

## Interpretation

By analyzing the centroids and plots, we can name our segments:
1.  **Cluster 0**: *Conservative Spenders* (Low-Mid Income, Low Spend)
2.  **Cluster 1**: *Standard Customers* (Middle Class, Moderate Spend)
3.  **Cluster 2**: *Target Group* (High Income, High Spend)
4.  **Cluster 3**: *Careful Wealthy* (High Income, Low Spend)
5.  **Cluster 4**: *Impulse Youth* (Young, Low Income, High Spend)

(Note: Cluster indices may vary slightly due to random initialization)